# LASSO Regression for Feature Selection

Practice activity from Microsoft *Foundations of AI and Machine Learning* — Module: AI/ML Algorithms and Techniques.

Goal: apply LASSO (L1 regularization) to a small dataset, sweep the regularization strength `alpha`, and read off which features survive.

## 2. Imports

In [1]:
import pandas as pd
from sklearn.linear_model import Lasso
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score

## 3. Load and prepare the data

In [2]:
data = {
    'StudyHours':    [1,  2,  3,  4,  5,  6,  7,  8,  9, 10],
    'PrevExamScore': [30, 40, 45, 50, 60, 65, 70, 75, 80, 85],
    'Pass':          [0,  0,  0,  0,  0,  1,  1,  1,  1,  1],
}
df = pd.DataFrame(data)

X = df[['StudyHours', 'PrevExamScore']]
y = df['Pass']
df

,StudyHours,PrevExamScore,Pass
0,1,30,0
1,2,40,0
2,3,45,0
3,4,50,0
4,5,60,0
5,6,65,1
6,7,70,1
7,8,75,1
8,9,80,1
9,10,85,1


## 4. Train/test split

In [3]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

## 5. Apply LASSO

LASSO adds an **L1 penalty** to the loss: the model now minimizes squared error *plus* `alpha · sum(|coefficients|)`. As `alpha` grows, the penalty pushes small coefficients exactly to zero — that's the feature-selection effect.

In [4]:
lasso_model = Lasso(alpha=0.1)
lasso_model.fit(X_train, y_train)
y_pred = lasso_model.predict(X_test)
r2 = r2_score(y_test, y_pred)

print(f'R-squared: {r2:.4f}')

R-squared: 0.9998


## 6. Inspect the coefficients

A coefficient of exactly `0.0` means LASSO has dropped the feature from the model.

In [5]:
print(f'LASSO Coefficients: {lasso_model.coef_}')
for name, coef in zip(X.columns, lasso_model.coef_):
    status = 'kept' if coef != 0 else 'dropped (shrunk to 0)'
    print(f'  {name:14s} coef={coef:+.6f}  [{status}]')

LASSO Coefficients: [0.         0.02463636]
  StudyHours     coef=+0.000000  [dropped (shrunk to 0)]
  PrevExamScore  coef=+0.024636  [kept]


## 7. Tune the regularization parameter

Sweep `alpha` from small to large and watch what happens to the coefficients and the R-squared score.

In [6]:
print(f"{'alpha':>8s}  {'R-squared':>12s}  coefficients")
for alpha in [0.01, 0.05, 0.1, 0.5, 1.0]:
    m = Lasso(alpha=alpha)
    m.fit(X_train, y_train)
    pred = m.predict(X_test)
    print(f'{alpha:>8.2f}  {r2_score(y_test, pred):>12.4f}  {m.coef_}')

   alpha     R-squared  coefficients
    0.01        0.9981  [0.08153909 0.01180619]
    0.05        0.9999  [0.         0.02481818]
    0.10        0.9998  [0.         0.02463636]
    0.50        0.9947  [0.         0.02318182]
    1.00        0.9788  [0.         0.02136364]


## Key takeaways

- LASSO's L1 penalty does **two things at once**: shrinks coefficients (regularization) and sets some to exactly zero (feature selection).
- **Low `alpha`** keeps most features but barely regularizes — overfitting risk on noisy data.
- **High `alpha`** is aggressive: at `alpha=1.0` here, only `PrevExamScore` survives and R² drops slightly as the surviving coefficient gets shrunk further.
- `StudyHours` and `PrevExamScore` grow together perfectly in this toy dataset, so LASSO keeps one and discards the other. With more features, LASSO is a quick way to pre-rank them by relevance.
- Choosing `alpha` is itself a tuning problem — `GridSearchCV` with R² scoring is the usual approach on real datasets.